# S3.7 — DataFrame Operations: select, filter, where, distinct, limit, sample
**Date completed:** October 2026  
**Status:** Completed
**Dataset:** samples.tpch — real production-scale data
**Interview covered:** Q8 — Transformations and Actions

In [0]:
# ============================================================
# S3.7 — Cell 2
# Load samples.tpch Dataset
#
# Goal:
# 1. Load orders and customer tables.
# 2. Inspect schemas before applying transformations.
# 3. Preview sample records.
# 4. Validate that both datasets contain data.
# 5. Establish the primary practice DataFrames for S3.7.
#
# Dataset:
# samples.tpch — TPC-H benchmark dataset
#
# Environment:
# Databricks Serverless
# ============================================================

print("=== LOADING samples.tpch DATASET ===")
print()

# ------------------------------------------------------------
# Step 1 — Load orders table
#
# spark.table() returns a DataFrame representing a
# registered table.
#
# Spark DataFrame transformations are generally lazy.
# Actions such as count() and show() trigger execution
# when Spark needs to produce a result.
# ------------------------------------------------------------

df_orders = spark.table(
    "samples.tpch.orders"
)

print("=== ORDERS TABLE ===")
print("\n--- Orders schema ---")

df_orders.printSchema()

# ------------------------------------------------------------
# Step 2 — Count orders
#
# count() is an ACTION.
#
# Spark must execute the required plan and return the
# total number of rows.
# ------------------------------------------------------------

orders_count = df_orders.count()

print(f"\nTotal order rows: {orders_count:,}")

# ------------------------------------------------------------
# Step 3 — Preview orders
#
# show(5) displays five rows.
#
# show() is an ACTION because Spark must retrieve records
# for display.
#
# Important:
# Without orderBy(), these should not be interpreted as
# the business-defined "first five" orders.
# ------------------------------------------------------------

print("\n--- Sample order rows ---")

df_orders.show(
    5,
    truncate=False
)

# ------------------------------------------------------------
# Step 4 — Load customer table
#
# This table will also be used in later S3.7 examples.
# ------------------------------------------------------------

df_customer = spark.table(
    "samples.tpch.customer"
)

print("\n=== CUSTOMER TABLE ===")

print("\n--- Customer schema ---")

df_customer.printSchema()

# ------------------------------------------------------------
# Step 5 — Count customers
#
# count() again triggers Spark execution.
# ------------------------------------------------------------

customer_count = df_customer.count()

print(f"\nTotal customer rows: {customer_count:,}")

# ------------------------------------------------------------
# Step 6 — Preview customers
# ------------------------------------------------------------

print("\n--- Sample customer rows ---")

df_customer.show(
    5,
    truncate=False
)


# ------------------------------------------------------------
# Step 7 — Basic validation
#
# assert checks whether an expected condition is true.
#
# These assertions confirm that both tables contain records.
#
# assert does not modify the DataFrames.
# ------------------------------------------------------------

print("\n=== BASIC VALIDATION ===")

assert orders_count > 0, (
    "Orders table is empty"
)

assert customer_count > 0, (
    "Customer table is empty"
)

print("Orders table loaded successfully.")
print("Customer table loaded successfully.")


# ------------------------------------------------------------
# Step 8 — Final learning summary
# ------------------------------------------------------------

print("\n=== FINAL LEARNING SUMMARY ===")

print("""
spark.table():
- Loads a registered table as a Spark DataFrame.

printSchema():
- Displays the DataFrame schema.
- Shows column names, data types, and nullability.

count():
- Returns the total number of rows.
- It is an ACTION.

show():
- Displays DataFrame rows.
- It is also an ACTION.

Important:
- Spark transformations are generally evaluated lazily.
- Actions require Spark to produce a result.
- show(5) does not mean the business-defined first five rows
  unless an explicit orderBy() is applied.

Practice DataFrames created:
- df_orders
- df_customer

These DataFrames will be used in upcoming S3.7 practicals:

- select()
- filter()
- where()
- distinct()
- limit()
- sample()
""")

print("=== CELL 2 COMPLETED ===")

In [0]:
# ============================================================
# S3.7 — Cell 3
# Core DataFrame Operations
#
# Goal:
# 1. Use select() to choose and rename columns.
# 2. Use filter() and where() to restrict rows.
# 3. Apply multiple filtering conditions.
# 4. Use distinct() to remove duplicate rows/values.
# 5. Use limit() to restrict the number of returned rows.
# 6. Use sample() for approximate random sampling.
# 7. Reinforce transformations vs actions.
#
# Dataset:
# samples.tpch.orders
#
# Environment:
# Databricks Serverless
#
# IMPORTANT:
# This learning cell intentionally calls several actions
# such as count() and show(). Each action can trigger Spark work.
# ============================================================

import pyspark.sql.functions as F

print("=== S3.7: CORE DATAFRAME OPERATIONS ===")
print()


# ------------------------------------------------------------
# Step 1 — Load orders table
#
# spark.table() returns a DataFrame representing the
# registered table.
# ------------------------------------------------------------

df_orders = spark.table("samples.tpch.orders")

print("Orders DataFrame loaded.")

orders_count = df_orders.count()

print(f"Total order rows: {orders_count:,}")
print()

# ------------------------------------------------------------
# Step 2 — select(): Choose Specific Columns
#
# select() is a TRANSFORMATION.
#
# It creates a new DataFrame containing only the selected
# columns or expressions.
#
# Spark's optimizer can apply column pruning when the
# underlying data source supports it.
# ------------------------------------------------------------

print("=== STEP 2: select() ===")

# Method 1 — Select using string column names

df_select1 = df_orders.select(
    "o_orderkey",
    "o_custkey",
    "o_orderstatus",
    "o_totalprice",
    "o_orderdate"
)

print("\n--- Method 1: String column names ---")

df_select1.show(3, truncate = False)


# Method 2 — Select using F.col()
#
# F.col() is useful when we need:
# - aliases
# - expressions
# - calculations
# - conditional logic
# - function calls

df_select2 = df_orders.select(

    F.col("o_orderkey").alias("order_id"),
    F.col("o_custkey").alias("customer_id"),
    F.col("o_orderstatus").alias("status"),
    F.col("o_totalprice").alias("total_price"),
    F.col("o_orderdate").alias("order_date"))

print("\n--- Method 2: F.col() with aliases ---")
df_select2.show(3, truncate=False)

# Step 3 — filter() and where()
# filter() and where() are equivalent DataFrame operations.
# Both return rows that satisfy a Boolean condition.
# Conditions can be written using:
# 1. Spark Column expressions
# 2. SQL expression strings
# ------------------------------------------------------------

print("=== STEP 3: filter() vs where() ===")

# filter() with Column expression

df_filter = df_orders.filter(
    F.col("o_orderstatus") == "F"
)

# where() with the same Column expression

df_where = df_orders.where(
    F.col("o_orderstatus") == "F"
)

# filter() using SQL expression syntax

df_sql_filter = df_orders.filter("o_orderstatus = 'F'")

# Execute counts

filter_count = df_filter.count()
where_count = df_where.count()
sql_filter_count = df_sql_filter.count()

print(f"filter() count     : {filter_count:,}")
print(f"where() count      : {where_count:,}")
print(f"SQL string count   : {sql_filter_count:,}")

# Validate that all three approaches return
# the same number of records.

assert filter_count == where_count, ("filter() and where() counts do not match")

assert filter_count == sql_filter_count, ("Column expression and SQL expression counts do not match")

print("Validation passed: all three approaches returned the same count.")
print()

# ------------------------------------------------------------
# Step 4 — Multiple Filter Conditions
#
# PySpark Boolean operators:
#
# &  → AND
# |  → OR
# ~  → NOT
#
# IMPORTANT:
# Use parentheses around individual conditions.
# ------------------------------------------------------------

print("=== STEP 4: MULTIPLE FILTER CONDITIONS ===")

# AND condition
#
# Status must be F
# AND
# total price must be greater than 100,000.

df_and = df_orders.filter(
    (F.col("o_orderstatus") == "F") & (F.col("o_totalprice") > 100000))

# OR condition
#
# Status can be either F or O.

df_or = df_orders.filter(
    (F.col("o_orderstatus") == "F") | (F.col("o_orderstatus") == "O"))

# NOT condition using ~
#
# Exclude rows where status is P.

df_not = df_orders.filter(~(F.col("o_orderstatus") == "P"))


and_count = df_and.count()
or_count = df_or.count()
not_count = df_not.count()

print(f"AND — F and price > 100K : {and_count:,}")
print(f"OR  — F or O             : {or_count:,}")
print(f"NOT — status is not P    : {not_count:,}")
print()

# ------------------------------------------------------------
# Step 5 — Useful Filtering Helpers
#
# isin()
#     Checks whether a value belongs to a list of values.
#
# between()
#     Checks whether a value falls inside an inclusive range.
# ------------------------------------------------------------

print("=== STEP 5: isin() AND between() ===")

# isin()
#
# Equivalent business requirement:
# status IN ('F', 'O')

df_isin = df_orders.filter(F.col("o_orderstatus").isin("F","O"))

isin_count = df_isin.count()

print(f"isin('F', 'O') rows : " f"{isin_count:,}")

# between()
#
# between(lower, upper) is inclusive.

df_between = df_orders.filter(F.col("o_totalprice").between(100000,200000))

between_count = df_between.count()

print(f"Price 100K-200K rows    : " f"{between_count:,}")

# OR result and isin result should match.

assert or_count == isin_count, ("OR condition and isin() produced different counts")

print("Validation passed: OR and isin() returned the same count.")
print()

# ------------------------------------------------------------
# Step 6 — distinct(): Remove Duplicate Rows
#
# distinct() returns unique rows based on ALL columns
# currently present in the DataFrame.
#
# Here we first select only o_orderstatus.
#
# Therefore distinct() returns the unique order statuses.
#
# distinct() commonly requires data movement/shuffling
# because duplicate values may exist across partitions.
# ------------------------------------------------------------

print("=== STEP 6: distinct() ===")

df_distinct_status = (df_orders.select("o_orderstatus").distinct())

print("\nDistinct order statuses:")

df_distinct_status.show(truncate = False)

distinct_status_count = (df_distinct_status.count())

print(f"Number of distinct statuses: " f"{distinct_status_count}")
print()

# ------------------------------------------------------------
# Step 7 — limit(): Restrict Number of Rows
#
# limit(N) creates a DataFrame containing at most N rows.
#
# IMPORTANT:
# Without orderBy(), Spark does NOT provide a business-defined
# meaning of "first N rows".
#
# limit() is useful for:
# - inspection
# - development
# - testing
# - small previews
# ------------------------------------------------------------

print("=== STEP 7: limit() ===")
df_limit = df_orders.limit(10)
limit_count = df_limit.count()

print(f"Rows returned after limit(10): "f"{limit_count}")

df_limit.select(
    "o_orderkey",
    "o_orderstatus",
    "o_totalprice"
).show(10, truncate=False)

print("""
Important:
limit(10) means return at most 10 rows.
It does NOT mean the business-defined first 10 orders.
For deterministic business ordering, use orderBy()
before applying limit().""")

# ------------------------------------------------------------
# Step 8 — Deterministic Top-N Example
#
# If the requirement is:
#
# "Show the 10 highest-value orders"
#
# We must define the ordering explicitly.
# ------------------------------------------------------------
print("=== STEP 8: ORDERED LIMIT EXAMPLE ===")

df_top10_orders = (df_orders.orderBy(F.col("o_totalprice").desc()).limit(10))

df_top10_orders.select(
    "o_orderkey",
    "o_orderstatus",
    "o_totalprice"
).show(10, truncate = False)
print()

# ------------------------------------------------------------
# Step 9 — sample(): Approximate Random Sampling
#
# Syntax:
#
# DataFrame.sample(
#     withReplacement,
#     fraction,
#     seed
# )
#
# fraction=0.001
#
# means approximately:
#
# 0.001 × 100 = 0.1%
#
# The resulting number of rows is NOT guaranteed
# to be exactly 0.1% of the source.
# ------------------------------------------------------------

print("=== STEP 9: sample() ===")


df_sample = df_orders.sample(
    withReplacement=False,
    fraction=0.001,
    seed=42
)

sample_count = df_sample.count()

expected_approx_sample = (orders_count * 0.001)

print(f"Source rows   : "f"{orders_count:,}")

print(f"Requested sample fraction: "f"0.001 = 0.1%")

print(f"Approximate expected rows: "f"{expected_approx_sample:,.0f}")

print(f"Actual sampled rows  : "f"{sample_count:,}")

print("""
Important:
- sample() performs approximate random sampling.
- fraction is a probability, not an exact row-count request.
- withReplacement=False means a selected row is not repeatedly
  sampled within the same sampling operation.
- seed=42 supports reproducible sampling for the same
  input and sampling setup.
""")

# ------------------------------------------------------------
# Step 10 — Transformations vs Actions
#
# Transformations create new DataFrames.
#
# Actions require Spark to produce a result.
# ------------------------------------------------------------

print("=== STEP 10: TRANSFORMATIONS VS ACTIONS ===")


print("""
TRANSFORMATIONS USED IN THIS CELL:

select()
filter()
where()
distinct()
limit()
sample()
orderBy()

These return new DataFrames.

They participate in Spark's lazy execution model.


ACTIONS USED IN THIS CELL:

count()
show()

These require Spark to execute work and return a result.

Important:
This learning cell calls several actions intentionally,
so Databricks may execute multiple Spark queries/jobs.
""")


# ------------------------------------------------------------
# Step 11 — Final Learning Summary
# ------------------------------------------------------------

print("=== FINAL LEARNING SUMMARY ===")


print("""
select():
- Chooses columns or expressions.
- Can rename columns using alias().

filter():
- Keeps rows satisfying a condition.

where():
- Equivalent to filter().

Boolean filtering:
- &  means AND
- |  means OR
- ~  means NOT

isin():
- Checks whether a value belongs to a supplied set of values.

between():
- Checks an inclusive range.

distinct():
- Removes duplicate rows based on the selected columns.
- May require a shuffle.

limit():
- Returns at most N rows.
- Does not define business ordering.

sample():
- Performs approximate random sampling.
- fraction is not an exact row-count guarantee.
- seed helps make sampling reproducible.

Core Spark Principle:
Transformations build the execution plan.
Actions require Spark to execute that plan.
""")


print("=== CELL 3 COMPLETED ===")


# S3.7 — DataFrame Operations: Final Key Takeaways

## 1. select()

`select()` chooses columns or expressions from a DataFrame.

### Select columns by name

```python
df.select(
    "o_orderkey",
    "o_custkey",
    "o_totalprice"
)
```

### Select using col()

```python
df.select(
    F.col("o_orderkey"),
    F.col("o_totalprice")
)
```

### Rename using alias()

```python
df.select(
    F.col("o_orderkey").alias("order_id"),
    F.col("o_totalprice").alias("total_price")
)
```

### When to use

Use `select()` when:

- Only specific columns are required.
- Columns need to be renamed.
- Expressions or calculations are required.
- Unnecessary columns should be excluded from downstream processing.

### Performance consideration

Spark's optimizer can apply **column pruning** so that unnecessary
columns are avoided where the underlying data source and execution plan
support it.

`select()` is a transformation.

---

# 2. filter() vs where()

Both perform the same DataFrame filtering operation.

### filter()

```python
df.filter(
    F.col("o_orderstatus") == "F"
)
```

### where()

```python
df.where(
    F.col("o_orderstatus") == "F"
)
```

### SQL expression syntax

```python
df.filter(
    "o_orderstatus = 'F'"
)
```

### Important

`filter()` and `where()` have equivalent filtering semantics.

They differ mainly in naming/style:

```text
filter() → common DataFrame/Python-style usage
where()  → familiar to SQL developers
```

Both are transformations.

---

# 3. Boolean Operators

| Operator | Meaning | Example |
|---|---|---|
| `&` | AND | `(status == "F") & (price > 100000)` |
| `\|` | OR | `(status == "F") \| (status == "O")` |
| `~` | NOT | `~(status == "P")` |
| `isin()` | Value belongs to a set | `status.isin("F", "O")` |
| `between()` | Inclusive range | `price.between(100000, 200000)` |

## AND

```python
df.filter(
    (F.col("o_orderstatus") == "F")
    &
    (F.col("o_totalprice") > 100000)
)
```

Both conditions must be true.

## OR

```python
df.filter(
    (F.col("o_orderstatus") == "F")
    |
    (F.col("o_orderstatus") == "O")
)
```

At least one condition must be true.

## NOT

```python
df.filter(
    ~(F.col("o_orderstatus") == "P")
)
```

## isin()

Instead of:

```python
(status == "F") | (status == "O")
```

we can write:

```python
F.col("o_orderstatus").isin(
    "F",
    "O"
)
```

## between()

```python
F.col("o_totalprice").between(
    100000,
    200000
)
```

`between()` is inclusive of both boundaries.

### Important

Use parentheses around individual Column conditions.

---

# 4. distinct()

`distinct()` removes duplicate rows based on all columns
currently present in the DataFrame.

Example:

```python
df_orders.select(
    "o_orderstatus"
).distinct()
```

This returns unique order-status values.

If we use:

```python
df_orders.distinct()
```

Spark compares the complete row across all columns.

### Important distinction

```text
select("status").distinct()
```

means:

```text
unique status values
```

while:

```text
df.distinct()
```

means:

```text
unique complete rows
```

### Performance consideration

`distinct()` commonly requires data movement between partitions
to identify duplicates across the dataset.

Therefore, it may introduce a shuffle and can be expensive
on large datasets.

Use it only when deduplication is actually required.

`distinct()` is a transformation.

---

# 5. limit()

`limit(N)` returns at most N rows.

Example:

```python
df.limit(10)
```

### Important

This does NOT mean:

```text
the business-defined first 10 rows
```

without an explicit ordering.

Example:

```python
df.limit(10)
```

means:

```text
Return at most 10 rows.
```

It does not define which rows are logically first.

### Typical uses

- Data inspection
- Development
- Debugging
- Small previews
- Testing

`limit()` is a transformation.

---

# 6. orderBy() + limit() — Top-N Pattern

If the business requirement is:

> Show the 10 highest-value orders.

Use:

```python
(
    df_orders
    .orderBy(
        F.col("o_totalprice").desc()
    )
    .limit(10)
)
```

This is different from:

```python
df_orders.limit(10)
```

### Easy way to remember

```text
limit(10)
→ Any at-most 10 rows according to the execution result

orderBy(...) + limit(10)
→ Top 10 according to the defined ordering
```

`orderBy()` is a transformation and generally requires ordering
data across partitions, which can involve a shuffle.

---

# 7. sample()

`sample()` creates an approximate random sample of a DataFrame.

Syntax:

```python
df.sample(
    withReplacement=False,
    fraction=0.001,
    seed=42
)
```

## fraction

```text
0.001 = 0.1%
```

This means rows are sampled probabilistically.

It does NOT mean Spark will return exactly 0.1% of the rows.

For example:

```text
Source = 7,500,000 rows

Approximate 0.1%
≈ 7,500 rows
```

The actual result may be somewhat higher or lower.

## withReplacement=False

A selected row is not repeatedly sampled within the same
sampling operation.

## seed

```python
seed=42
```

supports reproducible sampling when the input and sampling
setup remain consistent.

### Typical uses

- Exploratory analysis
- Development with large datasets
- Statistical analysis
- Model-development datasets
- Quick testing on representative subsets

`sample()` is a transformation.

---

# 8. limit() vs sample()

| Feature | limit() | sample() |
|---|---|---|
| Selection | Restricts row count | Random/probabilistic selection |
| Number of rows | At most N | Approximate fraction |
| Random | No sampling semantics | Yes |
| Business ordering | Not guaranteed without orderBy() | Not applicable |
| Reproducibility | Do not rely on row identity without ordering | Seed helps under stable input/setup |
| Typical use | Preview/testing | Sampling/analysis |

### Easy way to remember

```text
Need only a small preview?
→ limit()

Need a random subset?
→ sample()
```

---

# 9. Transformations vs Actions

## Transformations

Transformations create a new DataFrame or logical plan.

Examples used in S3.7:

```text
select()
filter()
where()
distinct()
limit()
sample()
orderBy()
```

Transformations participate in Spark's lazy evaluation model.

Example:

```python
df_filtered = df_orders.filter(
    F.col("o_orderstatus") == "F"
)
```

Spark can build the logical plan without immediately producing
all result rows.

---

## Actions

Actions require Spark to produce a result.

Examples:

```text
count()
show()
collect()
first()
take()
```

Example:

```python
df_filtered.count()
```

requires Spark to execute the required work and return the count.

Writing operations such as:

```python
df.write.parquet(...)
df.write.save(...)
df.write.saveAsTable(...)
```

also cause Spark to execute work and persist results.

### Important distinction

```python
df.write
```

by itself is only the DataFrameWriter interface.

A write operation such as:

```python
df.write.parquet(...)
```

actually executes the write.

---

# 10. Lazy Evaluation

One of the most important Spark concepts:

```text
Read DataFrame
     ↓
select()
     ↓
filter()
     ↓
orderBy()
     ↓
limit()
```

Spark builds an execution plan.

Then:

```python
show()
```

or:

```python
count()
```

requires Spark to execute the required plan.

### Easy way to remember

```text
Transformation
→ What should Spark do?

Action
→ Execute enough of the plan to return/persist a result.
```

---

# 11. Transformation Flow Example

```python
df_result = (
    df_orders

    .select(
        "o_orderkey",
        "o_orderstatus",
        "o_totalprice"
    )

    .filter(
        F.col("o_orderstatus") == "F"
    )

    .filter(
        F.col("o_totalprice") > 100000
    )

    .orderBy(
        F.col("o_totalprice").desc()
    )

    .limit(10)
)
```

Everything above builds the DataFrame transformation plan.

Then:

```python
df_result.show()
```

requires Spark to produce the result.

---

# 12. Performance Awareness

## select()

Selecting only required columns can help Spark reduce unnecessary
data processing through optimizer techniques such as column pruning.

## filter()

Filtering early can reduce the amount of data processed by
subsequent operations.

Depending on the data source, Spark may also apply predicate
pushdown or other optimizations.

## distinct()

Potentially expensive because duplicate detection across
partitions may require a shuffle.

## orderBy()

Global ordering can be expensive because records may need
to be redistributed and sorted.

## sample()

Useful when the entire dataset is unnecessary for development
or exploratory work.

---

# 13. Quick Decision Guide

| Requirement | Operation |
|---|---|
| Choose columns | `select()` |
| Rename selected columns | `select()` + `alias()` |
| Keep matching rows | `filter()` / `where()` |
| Check several allowed values | `isin()` |
| Check an inclusive range | `between()` |
| Remove duplicate rows | `distinct()` |
| Preview at most N rows | `limit()` |
| Find Top N records | `orderBy()` + `limit()` |
| Take an approximate random subset | `sample()` |

---

# 14. Interview Questions and Answers

### Q1. What is the difference between select() and filter()?

`select()` controls columns.

`filter()` controls rows.

```text
select()
→ Horizontal choice of columns

filter()
→ Vertical restriction of rows
```

---

### Q2. What is the difference between filter() and where()?

They provide equivalent DataFrame filtering semantics.

The difference is primarily naming/style.

---

### Q3. What does distinct() do?

It removes duplicate rows based on every column currently
present in the DataFrame.

---

### Q4. Does distinct() cause a shuffle?

It commonly requires data movement between partitions so that
Spark can identify duplicates across the dataset.

Therefore it can introduce a shuffle and be expensive.

---

### Q5. What is the difference between limit() and sample()?

`limit()` restricts the result to at most N rows.

`sample()` probabilistically selects an approximate fraction
of the dataset.

---

### Q6. Does limit(10) mean the first 10 rows?

Not in a business-defined sense unless ordering has been specified.

For Top-N requirements:

```python
df.orderBy(...).limit(10)
```

---

### Q7. Does sample(0.1) guarantee exactly 10% of the records?

No.

It performs approximate random sampling.

The resulting number of records can vary around the expected fraction.

---

### Q8. Why use a seed with sample()?

A seed helps make sampling reproducible when the input and sampling
setup remain consistent.

---

### Q9. What is the difference between a transformation and an action?

A transformation defines a new DataFrame/logical operation.

An action requires Spark to produce a result.

Examples:

```text
Transformations:
select()
filter()
distinct()
sample()

Actions:
count()
show()
collect()
```

---

### Q10. Why is Spark called lazily evaluated?

Because Spark can build a sequence of transformations without
immediately producing the complete result.

Execution occurs when an action requires a result.

---

### Q11. Why should we avoid collect() on very large datasets?

`collect()` brings all resulting records to the driver.

For a very large DataFrame this can consume excessive driver
memory and may cause failure.

Use it only when the result is known to be small.

---

### Q12. What is column pruning?

An optimization where Spark reads/processes only required columns
when possible instead of unnecessary columns.

Selecting only required columns can enable this optimization.

---

# 15. Final S3.7 Understanding

```text
Need columns?
→ select()

Need rows matching conditions?
→ filter() / where()

Need AND / OR / NOT?
→ & / | / ~

Need values from a set?
→ isin()

Need an inclusive range?
→ between()

Need unique rows?
→ distinct()

Need a quick small result?
→ limit()

Need Top N?
→ orderBy() + limit()

Need random subset?
→ sample()
```

And remember the core Spark execution principle:

```text
TRANSFORMATIONS
Build the plan

        ↓

ACTIONS
Require Spark to execute work and return/persist a result
```

**S3.7 — Core DataFrame Operations practical completed.**